# Clase 04: Hadoop, Hive, Spark y minería de datos

**Objetivos de aprendizaje de hoy:**
- Explicar qué problema resuelve HDFS.
- Diferenciar Hadoop, Hive y Spark.
- Describir el proceso de minería de datos en 6 pasos.

**Duración estimada:** 100 minutos

Capítulo del manual asociado: [05 — El ecosistema Hadoop, Hive y Spark](../manual/05_hadoop_hive_spark.md)

## ¿Por qué no basta con "comprar un ordenador más grande"?

Arrancamos con el problema real, no con la solución: contar palabras en cientos de millones de documentos, o indexar todas las páginas web del mundo (el problema original de Google a finales de los 90). Ningún ordenador individual —por potente que sea— tiene memoria ni tiempo suficiente.

La solución no fue escalar *verticalmente* (un ordenador más grande), sino escalar *horizontalmente*: repartir el trabajo entre **muchos** ordenadores normales que colaboran como si fueran uno solo.

- **Nodo**: un ordenador individual dentro del grupo.
- **Clúster**: un conjunto de nodos trabajando juntos. Puede tener desde un puñado de nodos hasta miles.

## Las tres piezas del ecosistema

| Pieza | Qué es | Para qué sirve |
|---|---|---|
| **Hadoop** | Framework Java open source | Almacena y procesa datos repartidos entre nodos de un clúster |
| **HDFS** | Sistema de archivos de Hadoop | Parte archivos grandes en bloques, los reparte y los replica |
| **Hive** | Almacén de datos sobre Hadoop | Consultas tipo SQL sobre datos enormes (alta latencia, orientado a lectura) |
| **Spark** | Motor de procesamiento en memoria | Analítica casi en tiempo real, streaming, ML — mucho más rápido que ir a disco |

**Analogía de HDFS**: piensa en una guía telefónica gigante de todo EE.UU. En vez de guardarla entera en un sitio, HDFS la parte por inicial de apellido y reparte los trozos entre servidores distintos. Por defecto, además, **replica cada bloque en dos servidores adicionales** — si uno se cae, los datos siguen disponibles en otro sitio.

Spark, a diferencia de Hadoop "clásico", procesa en **memoria (RAM)** en vez de escribir constantemente a disco: por eso es mucho más rápido. Puede correr solo o encima de Hadoop, y sabe leer datos de HDFS, de Hive y de muchas otras fuentes.

## El proceso de minería de datos en 6 pasos

Toda esta infraestructura existe para poder *minar* datos: descubrir patrones útiles en grandes volúmenes. Es un proceso **iterativo** — se repite en ciclos:

1. **Fijar el objetivo** — qué preguntas quieres responder (con criterio de coste/beneficio).
2. **Seleccionar fuentes de datos** — de dónde vienen los datos.
3. **Preprocesar** — detectar atributos irrelevantes o erróneos.
4. **Transformar** — elegir el formato adecuado de almacenamiento.
5. **Minar** — elegir métodos de análisis / algoritmos de ML.
6. **Evaluar** — probar la capacidad predictiva y compartir resultados.

Después del paso 6, el ciclo vuelve a empezar: cada iteración informa a la siguiente.

## Demo en vivo: MapReduce de juguete en Python puro

No vamos a instalar Hadoop en Colab (no hace falta, y no está permitido en este curso). Pero sí podemos programar la idea que lo hace funcionar por dentro: **MapReduce**.

- **Map**: cada nodo procesa su propio trozo de datos de forma independiente → produce pares `(clave, valor)`.
- **Reduce**: se combinan todos los pares intermedios agrupándolos por clave → resultado final.

Vamos a contar palabras en una colección de "documentos", simulando el reparto entre nodos de un clúster.

In [ ]:
# Documentos de ejemplo. En un clúster real, HDFS repartiría
# cada uno (o trozos de cada uno) entre distintos nodos.
documentos = [
    "los datos son el nuevo petroleo de la economia digital",
    "el petroleo se refina los datos tambien se deben refinar",
    "un cientifico de datos refina datos para encontrar patrones",
    "los patrones escondidos en los datos cuentan una historia",
    "hadoop reparte los datos entre muchos nodos de un cluster",
    "spark procesa los datos en memoria mucho mas rapido que hadoop",
]

def map_palabras(texto):
    """Fase MAP: recibe un documento y devuelve pares (palabra, 1).
    Esto es justo lo que haría, de forma independiente, cada nodo
    con el trozo de datos que le tocó."""
    palabras = texto.split()
    return [(palabra, 1) for palabra in palabras]

def reduce_conteos(pares):
    """Fase REDUCE: combina TODOS los pares intermedios (vengan
    del nodo que vengan) en un único diccionario palabra->conteo."""
    conteos = {}
    for palabra, valor in pares:
        conteos[palabra] = conteos.get(palabra, 0) + valor
    return conteos

In [ ]:
# Simulamos el reparto entre 3 "nodos" del clúster usando
# slicing simple de listas (sin threading real: aquí lo importante
# es el concepto, no la concurrencia física).
N_NODOS = 3
tamano_trozo = len(documentos) // N_NODOS + 1
nodos = [documentos[i:i + tamano_trozo] for i in range(0, len(documentos), tamano_trozo)]

print(f"Documentos repartidos en {len(nodos)} nodos:")
for i, datos_nodo in enumerate(nodos):
    print(f"  Nodo {i}: {len(datos_nodo)} documento(s)")

# Fase Map: cada nodo procesa SOLO sus propios documentos
pares_intermedios = []
for i, datos_nodo in enumerate(nodos):
    pares_nodo = []
    for doc in datos_nodo:
        pares_nodo.extend(map_palabras(doc))
    print(f"  Nodo {i} produjo {len(pares_nodo)} pares intermedios")
    pares_intermedios.extend(pares_nodo)

# Fase Reduce: un único proceso combina todo
conteo_final = reduce_conteos(pares_intermedios)
top_5 = sorted(conteo_final.items(), key=lambda par: par[1], reverse=True)[:5]
print("\nTop 5 palabras más frecuentes:")
for palabra, veces in top_5:
    print(f"  {palabra}: {veces}")

Fíjate: cada nodo solo vio **su propio trozo** de documentos en la fase Map — nunca necesitó saber nada de los demás nodos. Solo al final, en la fase Reduce, se combinaron todos los resultados parciales. Esa es la idea que permite a Hadoop procesar petabytes: miles de nodos trabajando en paralelo sobre trozos distintos, y una fase final de combinación.

## Resumen de la clase

- Hadoop reparte almacenamiento y procesamiento entre los nodos de un clúster; HDFS es su sistema de archivos (bloques + replicación).
- Hive permite consultar datos de Hadoop con SQL (alta latencia, orientado a lectura). Spark procesa en memoria, mucho más rápido, y sirve para analítica casi en tiempo real y ML.
- El proceso de minería de datos tiene 6 pasos iterativos: objetivo → fuentes → preprocesar → transformar → minar → evaluar.
- MapReduce = Map (cada nodo procesa su trozo) + Reduce (se combinan todos los resultados).

**Para seguir**:
- Capítulo del manual: [05 — El ecosistema Hadoop, Hive y Spark](../manual/05_hadoop_hive_spark.md)
- Notebook de práctica: [clase04_practica_hadoop_hive_spark.ipynb](../notebooks_practica/clase04_practica_hadoop_hive_spark.ipynb)